In [ ]:
# ============================================================
# BirdCLEF+ 2026
# Abdullah Sheikh
# ============================================================

In [ ]:
import os
import random
import hashlib
import json
import csv
import time
import datetime
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import timm
from tqdm import tqdm

In [ ]:
# Paths
BASE_DIR       = Path('/kaggle/input/competitions/birdclef-2026')
TRAIN_AUDIO    = BASE_DIR / 'train_audio'
TRAIN_SC_DIR   = BASE_DIR / 'train_soundscapes'
TEST_DIR       = BASE_DIR / 'test_soundscapes'
META_CSV       = BASE_DIR / 'train.csv'
TAXONOMY_CSV   = BASE_DIR / 'taxonomy.csv'
SC_LABELS_CSV  = BASE_DIR / 'train_soundscapes_labels.csv'
SAMPLE_SUB     = BASE_DIR / 'sample_submission.csv'

# Audio config
SR          = 32000
DURATION    = 5
N_MELS      = 128
HOP_LENGTH  = 512
N_FFT       = 1024
FMIN        = 50
FMAX        = 14000

In [ ]:
df     = pd.read_csv(META_CSV)
tax    = pd.read_csv(TAXONOMY_CSV)
labels = pd.read_csv(SC_LABELS_CSV)
sub    = pd.read_csv(SAMPLE_SUB)

train_files = list(TRAIN_AUDIO.rglob('*.ogg'))
sc_files    = list(TRAIN_SC_DIR.glob('*.ogg'))

print(f'Training clips     : {len(train_files)}')
print(f'Train soundscapes  : {len(sc_files)}')
print(f'Species to predict : {sub.shape[1] - 1}')
print(f'Labeled SC windows : {len(labels)}')
print(f'Submission row ex  : {sub["row_id"].iloc[0]}')

In [ ]:
print(f'Total recordings   : {len(df)}')
print(f'Unique species     : {df["primary_label"].nunique()}')
print(f'Animal classes     : {df["class_name"].nunique()}')
print(f'Data sources       : {df["collection"].nunique()} ({", ".join(df["collection"].unique())})')
print()
print(df["class_name"].value_counts().to_string())

In [ ]:
train_species = set(df['primary_label'].astype(str).unique())
tax_species   = set(tax['primary_label'].astype(str).unique())
missing       = tax_species - train_species

missing_info  = tax[tax['primary_label'].astype(str).isin(missing)]

print(f'Species in taxonomy (must predict) : {len(tax_species)}')
print(f'Species with training audio        : {len(train_species)}')
print(f'Species with NO training data      : {len(missing)}')
print()
print(missing_info[['common_name', 'class_name']].value_counts('class_name').to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# How many recordings does each species have?
# We expect a long tail  a few species dominate, many are underrepresented
counts = df['primary_label'].value_counts()
axes[0].bar(range(len(counts)), counts.values, color='steelblue')
axes[0].axhline(counts.median(), color='red', linestyle='--', label=f'Median: {counts.median():.0f}')
axes[0].set_title('Recordings per Species')
axes[0].set_xlabel('Species (sorted by count)')
axes[0].set_ylabel('Number of recordings')
axes[0].legend()

# The dataset is overwhelmingly birds  other classes are tiny minorities
class_counts = df['class_name'].value_counts()
axes[1].bar(class_counts.index, class_counts.values, color='steelblue')
axes[1].set_title('Recordings per Animal Class')
axes[1].set_xlabel('Animal class')
axes[1].set_ylabel('Number of recordings')

plt.suptitle('Training Data Distribution', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
labels['species_list'] = labels['primary_label'].str.split(';')
labels['n_species']    = labels['species_list'].apply(len)

# test set is multilabel
labels['n_species'].value_counts().sort_index().plot(
    kind='bar', figsize=(10, 4), color='steelblue', edgecolor='white'
)
plt.title('How many species are active in each 5s window?')
plt.xlabel('Number of species')
plt.ylabel('Number of windows')
plt.tight_layout()
plt.show()

In [ ]:
# Fixed reference values for 'global_mean_std' normalization.
# Based on typical 32 kHz log-mel spectrograms (power_to_db with ref=1.0).
GLOBAL_MEL_MEAN = -40.0
GLOBAL_MEL_STD  = 20.0


def load_audio(path, offset=0.0):
    duration = CONFIG.get('clip_duration_s', DURATION)
    audio, _ = librosa.load(str(path), sr=SR, offset=offset, duration=duration)
    if len(audio) < SR * duration:
        audio = np.pad(audio, (0, SR * duration - len(audio)))
    return audio


def load_audio_random_crop(path):
    """Load a random 5s window from an audio file."""
    duration = CONFIG.get('clip_duration_s', DURATION)
    try:
        file_dur = librosa.get_duration(path=str(path))
    except Exception:
        return load_audio(path, offset=0.0)
    if file_dur <= duration:
        return load_audio(path, offset=0.0)
    offset = np.random.uniform(0.0, file_dur - duration)
    return load_audio(path, offset=offset)


def is_silent(audio, threshold=0.01):
    return np.max(np.abs(audio)) < threshold


def audio_to_melspec(audio):
    n_mels        = CONFIG.get('n_mels', N_MELS)
    fmin          = CONFIG.get('fmin', FMIN)
    fmax          = CONFIG.get('fmax', FMAX)
    normalization = CONFIG.get('normalization', 'per_clip_minmax')

    mel = librosa.feature.melspectrogram(
        y=audio, sr=SR, n_mels=n_mels,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        fmin=fmin, fmax=fmax,
    )

    if normalization == 'db_fixed_floor':
        # Fixed [-80, 0] dB window, scaled to [0, 1]  preserves absolute loudness
        mel_db = librosa.power_to_db(mel, ref=1.0)
        mel_db = np.clip(mel_db, -80.0, 0.0) / 80.0 + 1.0
    elif normalization == 'global_mean_std':
        mel_db = librosa.power_to_db(mel, ref=1.0)
        mel_db = (mel_db - GLOBAL_MEL_MEAN) / (GLOBAL_MEL_STD + 1e-6)
    else:  # 'per_clip_minmax' (original behaviour)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)

    return mel_db.astype(np.float32)

In [ ]:
# A mel-spectrogram is how we feed audio into a CNN 
# it converts sound into a 2D image of frequency vs time.
# Let's look at what different animals actually look like.

def load_and_convert(path):
    audio, _ = librosa.load(str(path), sr=SR, duration=DURATION)
    if len(audio) < SR * DURATION:
        audio = np.pad(audio, (0, SR * DURATION - len(audio)))
    mel    = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS,
                                             n_fft=N_FFT, hop_length=HOP_LENGTH,
                                             fmin=FMIN, fmax=FMAX)
    return librosa.power_to_db(mel, ref=np.max)

samples = {
    'Aves (Bird)'  : df[(df['class_name'] == 'Aves') & (df['rating'] >= 4)].iloc[0],
    'Amphibia'     : df[df['class_name'] == 'Amphibia'].iloc[0],
    'Insecta'      : df[df['class_name'] == 'Insecta'].iloc[0],
    'Mammalia'     : df[df['class_name'] == 'Mammalia'].iloc[0],
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (cls, row) in zip(axes, samples.items()):
    mel_db = load_and_convert(TRAIN_AUDIO / row['filename'])
    librosa.display.specshow(mel_db, sr=SR, hop_length=HOP_LENGTH,
                              x_axis='time', y_axis='mel', ax=ax)
    ax.set_title(f'{cls}\n{row["common_name"][:25]}', fontsize=9)
    ax.set_xlabel('Time (s)')

plt.suptitle('Mel-Spectrograms  What Each Animal Class Looks Like', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
clean_row   = df[(df['class_name'] == 'Aves') & (df['rating'] >= 4)].iloc[0]
clean_audio, _ = librosa.load(str(TRAIN_AUDIO / clean_row['filename']), sr=SR, duration=DURATION)

sc_file     = sorted(TRAIN_SC_DIR.glob('*.ogg'))[0]
sc_audio, _ = librosa.load(str(sc_file), sr=SR, duration=DURATION)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, audio, title in [
    (axes[0], clean_audio, f'Clean training clip\n{clean_row["common_name"]}'),
    (axes[1], sc_audio,    f'Field soundscape\n{sc_file.name[:35]}')
]:
    mel    = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH, fmin=FMIN, fmax=FMAX)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(mel_db, sr=SR, hop_length=HOP_LENGTH, x_axis='time', y_axis='mel', ax=ax)
    ax.set_title(title)

plt.suptitle('Domain Gap  Training vs Test Data', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
#  Experiment Control 
TRAINING_MODE = True    # set False at submission  skips all training cells
USE_GPU       = True    # set False at submission  forces CPU inference

DEVICE = 'cuda' if (USE_GPU and torch.cuda.is_available()) else 'cpu'

#  Paths 
SPEC_DIR    = Path('/kaggle/working/spectrograms')    # training clip .npy files
SC_SPEC_DIR = Path('/kaggle/working/sc_spectrograms') # soundscape window .npy files
MODEL_PATH = Path('/kaggle/working/best_model.pt')  # change to Kaggle Dataset path at submission

#  Label Mapping 
NUM_CLASSES    = len(tax)   # 234
species_list   = tax['primary_label'].astype(str).tolist()
species_to_idx = {sp: i for i, sp in enumerate(species_list)}

print(f'Device       : {DEVICE}')
print(f'Classes      : {NUM_CLASSES}')
print(f'Training mode: {TRAINING_MODE}')

#  Experiment CONFIG 
# Single source of truth for all experiments.
# Pass a modified copy of this dict to run_experiment() to run ablations.
CONFIG = {
    # data pipeline
    'crop_mode'              : 'first5',   # 'first5' | 'random5'
    'clip_duration_s'        : 5,
    'use_secondary_labels'   : False,
    'secondary_weight'       : 0.3,        # label value for secondary species
    'use_soundscape_labels'  : False,      # add train_soundscapes_labels windows to training
    'soundscape_sample_weight': 3.0,
    'rating_threshold'       : None,       # XC-only; None | float (e.g. 2.5)

    # mel-spectrogram
    'n_mels'        : 128,
    'fmin'          : 50,
    'fmax'          : 14000,
    'normalization' : 'per_clip_minmax',   # 'per_clip_minmax' | 'db_fixed_floor' | 'global_mean_std'

    # augmentation
    'augmentations' : [],                  # any subset of ['specaug', 'mixup']
    'mixup_alpha'   : 0.5,

    # model
    'head'     : 'gap',                    # 'gap' | 'sed'
    'backbone' : 'efficientnet_b0',

    # loss
    'loss'        : 'bce',                 # 'bce' | 'bce_pos_weight' | 'focal'
    'focal_gamma' : 2.0,

    # optimiser & schedule
    'optimizer'     : 'adam',             # 'adam' | 'adamw'
    'weight_decay'  : 0.0,
    'lr'            : 1e-3,
    'scheduler'     : 'none',             # 'none' | 'cosine_warmup'
    'warmup_epochs' : 1,

    # training
    'epochs'     : 10,
    'batch_size' : 64,
    'patience'   : 3,   # early stopping on val macro AUC
    'seed'       : 42,
}

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(CONFIG['seed'])

#  Valid Training Clips 
# Require >=2 samples per species so stratified split works without crashing.
species_counts = df['primary_label'].value_counts()
valid_species  = set(species_counts[species_counts >= 2].index.astype(str))
valid_df       = df[df['primary_label'].astype(str).isin(valid_species)].copy().reset_index(drop=True)

print(f'Total clips   : {len(df)}')
print(f'Valid clips   : {len(valid_df)}  ({len(df) - len(valid_df)} dropped  species with <2 clips)')
print(f'Valid species : {valid_df["primary_label"].nunique()}')

#  Soundscape Validation Split 
# Reserve 200 windows as the fixed held-out validation set for ALL experiments.
# The remaining windows are optionally added to training (CONFIG['use_soundscape_labels']).
VAL_SC_N = 200

rng          = np.random.default_rng(42)
sc_perm      = rng.permutation(len(labels))
val_sc_df    = labels.iloc[sc_perm[:VAL_SC_N]].reset_index(drop=True)
train_sc_df  = labels.iloc[sc_perm[VAL_SC_N:]].reset_index(drop=True)

print(f'\nSoundscape windows : {len(labels)} total')
print(f'  Held-out val     : {len(val_sc_df)}')
print(f'  Available train  : {len(train_sc_df)}')

In [ ]:
# Precompute training clip spectrograms once using default mel params
# (per_clip_minmax, 128 mels, first-5s crop). BirdDataset loads these
# automatically when crop_mode='first5' and normalization='per_clip_minmax'.
# Re-running is safe -- existing files are skipped.
if TRAINING_MODE:
    SPEC_DIR.mkdir(exist_ok=True)
    done, skipped = 0, 0
    for _, row in tqdm(valid_df.iterrows(), total=len(valid_df), desc='Precomputing train specs'):
        npy_path = SPEC_DIR / (row['filename'].replace('/', '_').replace('.ogg', '.npy'))
        if npy_path.exists():
            skipped += 1
            continue
        try:
            audio = load_audio(TRAIN_AUDIO / row['filename'])
            if is_silent(audio):
                continue
            np.save(npy_path, audio_to_melspec(audio))
            done += 1
        except Exception:
            pass
    print(f'Precomputed {done} new training spectrograms (skipped {skipped} existing)')
    print(f'Total cached: {len(list(SPEC_DIR.glob("*.npy")))}')


In [ ]:
# Precompute soundscape window spectrograms once  eliminates librosa from
# eval_on_soundscapes and SoundscapeWindowDataset at training time.
# Only runs when files are missing, so re-running the cell is safe.
if TRAINING_MODE:
    SC_SPEC_DIR.mkdir(exist_ok=True)
    done = 0
    for _, row in tqdm(labels.iterrows(), total=len(labels), desc='Precomputing SC specs'):
        sc_path = TRAIN_SC_DIR / row['filename']
        if not sc_path.exists():
            continue
        parts     = str(row['start']).split(':')
        start_sec = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
        npy_path  = SC_SPEC_DIR / f"{Path(row['filename']).stem}_{start_sec:06d}.npy"
        if npy_path.exists():
            continue
        audio, _ = librosa.load(str(sc_path), sr=SR, offset=float(start_sec), duration=DURATION)
        if len(audio) < SR * DURATION:
            audio = np.pad(audio, (0, SR * DURATION - len(audio)))
        np.save(npy_path, audio_to_melspec(audio))
        done += 1
    print(f'Precomputed {done} new soundscape spectrograms -> {SC_SPEC_DIR}')
    print(f'Total cached: {len(list(SC_SPEC_DIR.glob("*.npy")))}')


In [ ]:
def _apply_specaug(spec):
    """SpecAugment: 2 frequency masks + 2 time masks."""
    C, H, W = spec.shape
    spec = spec.copy()
    for _ in range(2):
        f  = np.random.randint(1, 25)
        f0 = np.random.randint(0, max(1, H - f))
        spec[:, f0:f0 + f, :] = 0.0
    for _ in range(2):
        t  = np.random.randint(1, 41)
        t0 = np.random.randint(0, max(1, W - t))
        spec[:, :, t0:t0 + t] = 0.0
    return spec


def _parse_secondary_labels(raw):
    """Parse secondary_labels from train.csv (handles list-string and space-separated formats)."""
    if not raw or str(raw) in ('nan', '[]', ''):
        return []
    raw = str(raw).strip("[] '\"")
    tokens = raw.replace("'", "").replace('"', '').replace(',', ' ').split()
    return [t.strip() for t in tokens if t.strip()]


class BirdDataset(Dataset):
    """Training clips from train_audio/. Supports fixed-first-5s and random-crop modes."""

    def __init__(self, df, is_train=True):
        self.is_train = is_train
        self.samples  = []

        rating_threshold = CONFIG.get('rating_threshold')

        for _, row in df.iterrows():
            # XC-only quality filter
            if rating_threshold is not None and str(row.get('collection', '')) == 'XC':
                try:
                    if float(row.get('rating', 0)) < rating_threshold:
                        continue
                except (ValueError, TypeError):
                    pass

            npy_path = SPEC_DIR / (row['filename'].replace('/', '_').replace('.ogg', '.npy'))
            # Use precomputed .npy only when mel params match what was precomputed
            can_use_npy = (
                npy_path.exists() and
                CONFIG.get('crop_mode')      == 'first5' and
                CONFIG.get('normalization')  == 'per_clip_minmax' and
                CONFIG.get('n_mels', N_MELS) == N_MELS
            )
            self.samples.append({
                'filepath'  : TRAIN_AUDIO / row['filename'],
                'npy_path'  : npy_path if can_use_npy else None,
                'primary'   : str(row['primary_label']),
                'secondary' : _parse_secondary_labels(row.get('secondary_labels', '')),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        # Load spectrogram
        if s['npy_path'] is not None:
            spec = np.load(s['npy_path'])
        elif self.is_train and CONFIG['crop_mode'] == 'random5':
            spec = audio_to_melspec(load_audio_random_crop(s['filepath']))
        else:
            spec = audio_to_melspec(load_audio(s['filepath']))

        spec = np.stack([spec, spec, spec], axis=0)   # 3-channel for pretrained backbone

        if self.is_train and 'specaug' in CONFIG['augmentations']:
            spec = _apply_specaug(spec)

        # Multi-hot target
        target = np.zeros(NUM_CLASSES, dtype=np.float32)
        if s['primary'] in species_to_idx:
            target[species_to_idx[s['primary']]] = 1.0
        if CONFIG['use_secondary_labels']:
            for sp in s['secondary']:
                if sp in species_to_idx:
                    target[species_to_idx[sp]] = max(
                        target[species_to_idx[sp]], CONFIG['secondary_weight']
                    )

        return torch.tensor(spec, dtype=torch.float32), torch.tensor(target, dtype=torch.float32)

In [ ]:
class SoundscapeWindowDataset(Dataset):
    """Labeled 5s windows from train_soundscapes  in-domain training data."""

    def __init__(self, sc_df):
        self.samples = []
        for _, row in sc_df.iterrows():
            sc_path = TRAIN_SC_DIR / row['filename']
            if not sc_path.exists():
                continue
            parts     = str(row['start']).split(':')
            start_sec = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
            npy_path  = SC_SPEC_DIR / f"{Path(row['filename']).stem}_{start_sec:06d}.npy"
            self.samples.append({
                'filepath'   : sc_path,
                'start_sec'  : float(start_sec),
                'species_ids': [s.strip() for s in str(row['primary_label']).split(';')],
                'npy_path'   : npy_path,
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        can_use_npy = (
            s['npy_path'].exists() and
            CONFIG.get('normalization', 'per_clip_minmax') == 'per_clip_minmax' and
            CONFIG.get('n_mels', N_MELS) == N_MELS
        )
        if can_use_npy:
            spec = np.load(s['npy_path'])
        else:
            audio, _ = librosa.load(str(s['filepath']), sr=SR, offset=s['start_sec'], duration=DURATION)
            if len(audio) < SR * DURATION:
                audio = np.pad(audio, (0, SR * DURATION - len(audio)))
            spec = audio_to_melspec(audio)

        spec = np.stack([spec, spec, spec], axis=0)

        if 'specaug' in CONFIG['augmentations']:
            spec = _apply_specaug(spec)

        target = np.zeros(NUM_CLASSES, dtype=np.float32)
        for sp in s['species_ids']:
            if sp in species_to_idx:
                target[species_to_idx[sp]] = 1.0

        return torch.tensor(spec, dtype=torch.float32), torch.tensor(target, dtype=torch.float32)


In [ ]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2 ** 32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

if TRAINING_MODE:
    g = torch.Generator()
    g.manual_seed(CONFIG['seed'])

    train_df, val_df = train_test_split(
        valid_df,
        test_size=0.1,
        stratify=valid_df['primary_label'],
        random_state=CONFIG['seed'],
    )

    train_ds = BirdDataset(train_df, is_train=True)
    val_ds   = BirdDataset(val_df,   is_train=False)

    if CONFIG['use_soundscape_labels']:
        sc_train_ds = SoundscapeWindowDataset(train_sc_df)
        train_ds    = ConcatDataset([train_ds, sc_train_ds])
        print(f'Added {len(sc_train_ds)} soundscape training windows to training set')

    train_loader = DataLoader(
        train_ds, batch_size=CONFIG['batch_size'],
        shuffle=True, num_workers=2, pin_memory=True,
        worker_init_fn=seed_worker, generator=g,
    )
    val_loader = DataLoader(
        val_ds, batch_size=CONFIG['batch_size'],
        shuffle=False, num_workers=2, pin_memory=True,
    )

    print(f'Train samples : {len(train_ds)}')
    print(f'Val samples   : {len(val_ds)}')
    print(f'Train batches : {len(train_loader)}')

In [ ]:
class SEDHead(nn.Module):
    """Attention-pooled Sound Event Detection head.
    Replaces global average pooling with learned per-time-step attention,
    which helps when calls are short events inside a longer window.
    """
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.att = nn.Linear(in_features, num_classes)  # attention weights
        self.cls = nn.Linear(in_features, num_classes)  # per-frame class scores

    def forward(self, x):
        # x: (B, C, H, W)  backbone feature map
        x = x.mean(dim=2)          # pool frequency axis -> (B, C, W)
        x = x.permute(0, 2, 1)     # -> (B, T, C)
        att = torch.sigmoid(self.att(x))             # (B, T, num_classes)
        cls = self.cls(x)                            # (B, T, num_classes)
        out = (att * cls).sum(dim=1) / (att.sum(dim=1) + 1e-7)
        return out  # (B, num_classes)


class BirdClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        backbone_name = CONFIG['backbone']

        if CONFIG['head'] == 'gap':
            self.backbone = timm.create_model(
                backbone_name, pretrained=True, num_classes=NUM_CLASSES
            )
            self.sed_head = None
        else:  # 'sed'
            self.backbone = timm.create_model(
                backbone_name, pretrained=True, num_classes=0, global_pool=''
            )
            self.sed_head = SEDHead(self.backbone.num_features, NUM_CLASSES)

    def forward(self, x):
        if self.sed_head is None:
            return self.backbone(x)
        return self.sed_head(self.backbone(x))


model = BirdClassifier().to(DEVICE)
print(f'Backbone : {CONFIG["backbone"]}')
print(f'Head     : {CONFIG["head"]}')
print(f'Params   : {sum(p.numel() for p in model.parameters()):,}')
print(f'Output   : {NUM_CLASSES} classes')

In [ ]:
def run_experiment(cfg, notes=''):
    """Train one experiment from cfg dict and append one row to results.csv.

    Returns (best_auc, per_class_auc_array, checkpoint_path).
    Temporarily patches the global CONFIG so all helpers pick up cfg values.
    """
    seed_everything(cfg['seed'])

    cfg_str  = json.dumps(cfg, sort_keys=True)
    cfg_hash = hashlib.md5(cfg_str.encode()).hexdigest()[:8]
    ckpt_dir  = Path('/kaggle/working/ckpts')
    ckpt_dir.mkdir(exist_ok=True)
    ckpt_path = ckpt_dir / f'{cfg_hash}.pt'

    print(f'\n{"=" * 60}')
    print(f'Config hash : {cfg_hash}')
    print(f'Notes       : {notes}')
    print(f'{"=" * 60}')

    # Patch global CONFIG so BirdDataset / audio helpers read the experiment values
    old_cfg = CONFIG.copy()
    CONFIG.update(cfg)

    try:
        # Rating filter
        exp_df = valid_df.copy()
        rt = cfg.get('rating_threshold')
        if rt is not None:
            xc_mask = exp_df['collection'] == 'XC'
            def _ok(r):
                try:
                    return float(r) >= rt
                except (ValueError, TypeError):
                    return True
            exp_df = exp_df[~xc_mask | exp_df['rating'].apply(_ok)].reset_index(drop=True)

        g = torch.Generator()
        g.manual_seed(cfg['seed'])
        tr_df, _ = train_test_split(
            exp_df, test_size=0.1,
            stratify=exp_df['primary_label'],
            random_state=cfg['seed'],
        )

        tr_ds = BirdDataset(tr_df, is_train=True)
        if cfg.get('use_soundscape_labels'):
            tr_ds = ConcatDataset([tr_ds, SoundscapeWindowDataset(train_sc_df)])

        tr_loader = DataLoader(
            tr_ds, batch_size=cfg['batch_size'], shuffle=True,
            num_workers=2, pin_memory=True,
            worker_init_fn=seed_worker, generator=g,
        )

        m      = BirdClassifier().to(DEVICE)
        crit   = get_loss_fn(tr_df)
        opt    = get_optimizer(m)
        sched  = get_scheduler(opt)
        scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

        best_auc, patience_count = 0.0, 0
        best_per_class = None
        best_epoch     = 0
        train_min      = 0.0

        for epoch in range(1, cfg['epochs'] + 1):
            t0 = time.time()
            train_one_epoch(m, tr_loader, crit, opt, scaler, epoch)
            if sched is not None:
                sched.step()
            val_auc, per_cls = eval_on_soundscapes(m, val_sc_df)
            train_min += (time.time() - t0) / 60
            print(f'  Epoch {epoch} | Val AUC: {val_auc:.4f}')

            if val_auc > best_auc:
                best_auc, patience_count = val_auc, 0
                best_per_class = per_cls.copy()
                best_epoch     = epoch
                torch.save(m.state_dict(), ckpt_path)
            else:
                patience_count += 1
                if patience_count >= cfg['patience']:
                    print(f'  Early stopping at epoch {epoch}')
                    break

        # Per-category AUC from best checkpoint
        category_auc = {}
        for cat, key in [('Aves', 'val_auc_aves'), ('Amphibia', 'val_auc_amphibia'),
                          ('Insecta', 'val_auc_insecta'), ('Mammalia', 'val_auc_mammalia')]:
            idxs = [
                species_to_idx[sp]
                for sp in tax[tax['class_name'] == cat]['primary_label'].astype(str)
                if sp in species_to_idx
                   and best_per_class is not None
                   and not np.isnan(best_per_class[species_to_idx[sp]])
            ]
            category_auc[key] = float(np.mean(best_per_class[idxs])) if idxs else float('nan')

        # Inference speed (single window on DEVICE)
        m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        m.eval()
        dummy = torch.randn(1, 3, cfg.get('n_mels', 128), 313).to(DEVICE)
        with torch.no_grad():
            _ = m(dummy)  # warmup
            t0 = time.time()
            for _ in range(50):
                _ = m(dummy)
            inf_ms = (time.time() - t0) / 50 * 1000

        # Append row to results.csv
        results_path = Path('/kaggle/working/results.csv')
        row = {
            'timestamp'         : datetime.datetime.now().isoformat(timespec='seconds'),
            'config_hash'       : cfg_hash,
            'best_epoch'        : best_epoch,
            'val_auc_macro'     : round(best_auc, 4),
            'val_auc_aves'      : round(category_auc['val_auc_aves'], 4),
            'val_auc_amphibia'  : round(category_auc['val_auc_amphibia'], 4),
            'val_auc_insecta'   : round(category_auc['val_auc_insecta'], 4),
            'val_auc_mammalia'  : round(category_auc['val_auc_mammalia'], 4),
            'train_min'         : round(train_min, 1),
            'inf_ms_per_window' : round(inf_ms, 1),
            'notes'             : notes,
            'config_json'       : cfg_str,
        }
        write_header = not results_path.exists()
        with open(results_path, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=list(row.keys()))
            if write_header:
                writer.writeheader()
            writer.writerow(row)

        print(f'\nDone | AUC={best_auc:.4f} | {train_min:.1f} min | saved -> {ckpt_path.name}')
        return best_auc, best_per_class, ckpt_path

    finally:
        CONFIG.clear()
        CONFIG.update(old_cfg)

In [ ]:
def get_loss_fn(df=None):
    """Build the loss function for the current CONFIG."""
    if CONFIG['loss'] == 'bce_pos_weight':
        if df is None:
            df = train_df
        pos_counts = np.zeros(NUM_CLASSES)
        for _, row in df.iterrows():
            sp = str(row['primary_label'])
            if sp in species_to_idx:
                pos_counts[species_to_idx[sp]] += 1
        weights = np.clip((len(df) - pos_counts) / (pos_counts + 1e-6), 1.0, 50.0)
        pw = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
        return nn.BCEWithLogitsLoss(pos_weight=pw)

    elif CONFIG['loss'] == 'focal':
        gamma  = CONFIG['focal_gamma']
        bce_fn = nn.BCEWithLogitsLoss(reduction='none')
        def focal_loss(logits, targets):
            bce   = bce_fn(logits, targets)
            probs = torch.sigmoid(logits)
            pt    = torch.where(targets >= 0.5, probs, 1 - probs)
            return ((1 - pt) ** gamma * bce).mean()
        return focal_loss

    return nn.BCEWithLogitsLoss()


def get_optimizer(model):
    if CONFIG['optimizer'] == 'adamw':
        return optim.AdamW(
            model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay']
        )
    return optim.Adam(model.parameters(), lr=CONFIG['lr'])


def get_scheduler(optimizer):
    if CONFIG['scheduler'] != 'cosine_warmup':
        return None
    warmup = CONFIG['warmup_epochs']
    total  = CONFIG['epochs']
    def lr_lambda(epoch):
        if epoch < warmup:
            return max(1e-4, epoch / warmup)
        progress = (epoch - warmup) / max(1, total - warmup)
        return max(1e-2, 0.5 * (1 + np.cos(np.pi * progress)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def eval_on_soundscapes(model, sc_df):
    """Evaluate on labeled soundscape windows. Returns (macro_auc, per_class_auc_array).
    per_class_auc_array has np.nan for classes with no positive labels in sc_df.
    """
    model.eval()
    all_probs, all_targets = [], []
    can_use_npy = (
        CONFIG.get('normalization', 'per_clip_minmax') == 'per_clip_minmax' and
        CONFIG.get('n_mels', N_MELS) == N_MELS
    )

    with torch.no_grad():
        for _, row in sc_df.iterrows():
            sc_path = TRAIN_SC_DIR / row['filename']
            if not sc_path.exists():
                continue
            parts     = str(row['start']).split(':')
            start_sec = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
            npy_path  = SC_SPEC_DIR / f"{Path(row['filename']).stem}_{start_sec:06d}.npy"
            if can_use_npy and npy_path.exists():
                spec = np.load(npy_path)
            else:
                audio, _ = librosa.load(str(sc_path), sr=SR, offset=float(start_sec), duration=DURATION)
                if len(audio) < SR * DURATION:
                    audio = np.pad(audio, (0, SR * DURATION - len(audio)))
                spec = audio_to_melspec(audio)

            spec = np.stack([spec, spec, spec], axis=0)
            spec = torch.tensor(spec, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            probs = torch.sigmoid(model(spec)).cpu().numpy()[0]

            target = np.zeros(NUM_CLASSES, dtype=np.float32)
            for sp in str(row['primary_label']).split(';'):
                sp = sp.strip()
                if sp in species_to_idx:
                    target[species_to_idx[sp]] = 1.0

            all_probs.append(probs)
            all_targets.append(target)

    probs_arr   = np.array(all_probs)
    targets_arr = np.array(all_targets)

    per_class_auc = np.full(NUM_CLASSES, np.nan)
    valid_scores  = []
    for i in range(NUM_CLASSES):
        if targets_arr[:, i].sum() > 0:
            score = roc_auc_score(targets_arr[:, i], probs_arr[:, i])
            per_class_auc[i] = score
            valid_scores.append(score)

    macro_auc = float(np.mean(valid_scores)) if valid_scores else 0.0
    return macro_auc, per_class_auc


def train_one_epoch(model, loader, criterion, optimizer, scaler, epoch):
    model.train()
    total_loss = 0

    for batch_idx, (specs, targets) in enumerate(loader):
        specs, targets = specs.to(DEVICE), targets.to(DEVICE)

        if 'mixup' in CONFIG['augmentations'] and np.random.random() < 0.5:
            lam     = float(np.random.beta(CONFIG['mixup_alpha'], CONFIG['mixup_alpha']))
            idx     = torch.randperm(specs.size(0), device=DEVICE)
            specs   = lam * specs   + (1 - lam) * specs[idx]
            targets = lam * targets + (1 - lam) * targets[idx]

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            logits = model(specs)
            loss   = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f'  Epoch {epoch} | Batch {batch_idx}/{len(loader)} | Loss: {loss.item():.4f}')

    return total_loss / len(loader)


#  Experiment 0  Baseline Training Run 
if TRAINING_MODE:
    criterion = get_loss_fn(train_df)
    optimizer = get_optimizer(model)
    scheduler = get_scheduler(optimizer)
    scaler    = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

    best_auc, patience_count = 0.0, 0
    history = []

    for epoch in range(1, CONFIG['epochs'] + 1):
        print(f'\nEpoch {epoch}/{CONFIG["epochs"]}')
        t0         = time.time()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, epoch)

        if scheduler is not None:
            scheduler.step()

        val_auc, _ = eval_on_soundscapes(model, val_sc_df)
        elapsed    = (time.time() - t0) / 60
        print(f'  Train Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f} | {elapsed:.1f} min')
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_auc': val_auc})

        if val_auc > best_auc:
            best_auc, patience_count = val_auc, 0
            torch.save(model.state_dict(), MODEL_PATH)
            print(f'  Best model saved (AUC {best_auc:.4f})')
        else:
            patience_count += 1
            if patience_count >= CONFIG['patience']:
                print(f'  Early stopping at epoch {epoch}')
                break

    # Load best checkpoint and show per-class breakdown
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    final_auc, per_class_auc = eval_on_soundscapes(model, val_sc_df)
    print(f'\nBest Val AUC (200-window holdout): {final_auc:.4f}')

    valid_idx = np.where(~np.isnan(per_class_auc))[0]
    if len(valid_idx) > 0:
        sorted_idx = valid_idx[np.argsort(per_class_auc[valid_idx])]
        print('\nBottom 20 classes by AUC:')
        for rank, i in enumerate(sorted_idx[:20]):
            print(f'  {rank+1:2d}. {species_list[i]:<15}  AUC={per_class_auc[i]:.3f}')

In [ ]:
# Sanity check on the full 1,478-window soundscape labels set.
# Includes the 200-window holdout, so this number is slightly optimistic
# compared to val_sc_df. The primary metric remains the 200-window holdout.
if TRAINING_MODE:
    full_auc, _ = eval_on_soundscapes(model, labels)
    print(f'Full soundscape labels ROC-AUC (all {len(labels)} windows): {full_auc:.4f}')

In [ ]:
# Capture baseline config once -- all ablation cells use {**BASE, ...} to
# derive single-factor variants. run_experiment() restores CONFIG after each
# run so BASE always reflects the original defaults.
BASE = dict(CONFIG)
print('BASE config captured. Starting ablation study.')
print(f'Results -> /kaggle/working/results.csv')


In [ ]:
# A0: Baseline re-run -- verifies precomputed specs work, establishes new baseline.
if TRAINING_MODE:
    auc_a0, _, _ = run_experiment(
        {**BASE},
        notes='A0: baseline re-run with precomputed specs'
    )


In [ ]:
# A1: Random crop -- does random 5s window during training beat fixed first-5s?
if TRAINING_MODE:
    auc_a1, _, _ = run_experiment(
        {**BASE, 'crop_mode': 'random5'},
        notes='A1: random5 crop vs first5'
    )


In [ ]:
# A2: SpecAugment -- time + frequency masking as regularisation.
if TRAINING_MODE:
    auc_a2, _, _ = run_experiment(
        {**BASE, 'augmentations': ['specaug']},
        notes='A2: specaug'
    )


In [ ]:
# A3: Soundscape labels -- add in-domain labeled windows to training.
# Most likely highest-impact lever: directly closes the domain gap.
if TRAINING_MODE:
    auc_a3, _, _ = run_experiment(
        {**BASE, 'use_soundscape_labels': True},
        notes='A3: soundscape labels in training'
    )


In [ ]:
# A4: EfficientNet-B1 -- more capacity (6.7M vs 4.3M params).
if TRAINING_MODE:
    auc_a4, _, _ = run_experiment(
        {**BASE, 'backbone': 'efficientnet_b1'},
        notes='A4: efficientnet_b1 backbone'
    )


In [ ]:
# A5: Cosine warmup LR -- 1-epoch warmup + cosine decay.
if TRAINING_MODE:
    auc_a5, _, _ = run_experiment(
        {**BASE, 'scheduler': 'cosine_warmup', 'warmup_epochs': 1},
        notes='A5: cosine_warmup schedule'
    )


In [ ]:
# A6: XC quality filter -- drop Xeno-canto clips rated below 3.0.
if TRAINING_MODE:
    auc_a6, _, _ = run_experiment(
        {**BASE, 'rating_threshold': 3.0},
        notes='A6: XC rating >= 3.0 quality filter'
    )


In [ ]:
# Quick-scan results after Sub-phase A.
# Read this before deciding which B factors to prioritise.
import pandas as pd
if TRAINING_MODE:
    res = pd.read_csv('/kaggle/working/results.csv')
    print(res[['notes','val_auc_macro','val_auc_aves','val_auc_amphibia',
               'val_auc_insecta','val_auc_mammalia','train_min']]
          .sort_values('val_auc_macro', ascending=False).to_string())


In [ ]:
# B1: Mixup -- linear interpolation between training samples.
if TRAINING_MODE:
    auc_b1, _, _ = run_experiment(
        {**BASE, 'augmentations': ['mixup']},
        notes='B1: mixup alpha=0.5'
    )


In [ ]:
# B2: SpecAugment + Mixup combined -- test if they are complementary.
if TRAINING_MODE:
    auc_b2, _, _ = run_experiment(
        {**BASE, 'augmentations': ['specaug', 'mixup']},
        notes='B2: specaug + mixup combined'
    )


In [ ]:
# B3: SED attention head -- learned per-time-step attention replaces GAP.
if TRAINING_MODE:
    auc_b3, _, _ = run_experiment(
        {**BASE, 'head': 'sed'},
        notes='B3: SED attention head vs GAP'
    )


In [ ]:
# B4: AdamW -- L2 regularisation via weight decay.
if TRAINING_MODE:
    auc_b4, _, _ = run_experiment(
        {**BASE, 'optimizer': 'adamw', 'weight_decay': 1e-4},
        notes='B4: adamw wd=1e-4'
    )


In [ ]:
# B5: Focal loss -- down-weights easy negatives, focuses on hard/rare classes.
if TRAINING_MODE:
    auc_b5, _, _ = run_experiment(
        {**BASE, 'loss': 'focal', 'focal_gamma': 2.0},
        notes='B5: focal loss gamma=2.0'
    )


In [ ]:
# B6: Secondary labels -- soft multi-species supervision (weight=0.3).
if TRAINING_MODE:
    auc_b6, _, _ = run_experiment(
        {**BASE, 'use_secondary_labels': True, 'secondary_weight': 0.3},
        notes='B6: secondary labels weight=0.3'
    )


In [ ]:
# B7: EfficientNet-B2 -- step above B1 in capacity (8.5M params).
# Run if A4/EfficientNet-B1 showed improvement over A0.
if TRAINING_MODE:
    auc_b7, _, _ = run_experiment(
        {**BASE, 'backbone': 'efficientnet_b2'},
        notes='B7: efficientnet_b2 backbone'
    )


In [ ]:
# Full results after Sub-phase B -- sorted by AUC to identify top factors.
import pandas as pd
if TRAINING_MODE:
    res = pd.read_csv('/kaggle/working/results.csv')
    print(res[['notes','val_auc_macro','val_auc_aves','val_auc_amphibia',
               'val_auc_insecta','val_auc_mammalia','train_min']]
          .sort_values('val_auc_macro', ascending=False).to_string())


In [ ]:
# C1: Best config combination -- update best_cfg with actual winners from A+B
# before running this cell. Train for 20 epochs with patience=5.
import shutil

if TRAINING_MODE:
    best_cfg = {
        **BASE,
        # Update these with actual winners from the A+B results table above.
        'use_soundscape_labels': True,
        'backbone'             : 'efficientnet_b1',
        'augmentations'        : ['specaug'],
        'crop_mode'            : 'random5',
        'scheduler'            : 'cosine_warmup',
        'warmup_epochs'        : 1,
        # Longer training for the best run
        'epochs'               : 20,
        'patience'             : 5,
    }
    best_auc, best_per_class, best_ckpt = run_experiment(
        best_cfg,
        notes='C1: best config combination -- 20 epochs'
    )
    shutil.copy(best_ckpt, '/kaggle/working/best_phase1.pt')
    print(f'best_phase1.pt saved. AUC = {best_auc:.4f}')


In [ ]:
# D1: Per-class AUC breakdown -- bottom 20 species + per-category summary.
# BirdClassifier reads backbone/head from CONFIG, so we must patch with
# best_cfg before construction, then restore BASE.
if TRAINING_MODE:
    CONFIG.update(best_cfg)
    best_model = BirdClassifier().to(DEVICE)
    CONFIG.update(BASE)

    best_model.load_state_dict(torch.load('/kaggle/working/best_phase1.pt', map_location=DEVICE))
    best_model.eval()

    _, per_class_auc = eval_on_soundscapes(best_model, val_sc_df)
    valid_idx  = np.where(~np.isnan(per_class_auc))[0]
    sorted_idx = valid_idx[np.argsort(per_class_auc[valid_idx])]

    print('=== Bottom 20 classes by AUC (best_phase1.pt, 200-window holdout) ===')
    for rank, i in enumerate(sorted_idx[:20]):
        sp   = species_list[i]
        rows = tax[tax['primary_label'] == sp]
        cat  = rows['class_name'].iloc[0] if len(rows) else 'unknown'
        name = rows['common_name'].iloc[0] if len(rows) else sp
        print(f'  {rank+1:2d}. {sp:<15}  {cat:<10}  AUC={per_class_auc[i]:.3f}  ({name})')

    print()
    print('=== Per-category macro AUC ===')
    for cat in ['Aves', 'Amphibia', 'Insecta', 'Mammalia']:
        idxs = [species_to_idx[sp]
                for sp in tax[tax['class_name'] == cat]['primary_label'].astype(str)
                if sp in species_to_idx and not np.isnan(per_class_auc[species_to_idx[sp]])]
        if idxs:
            print(f'  {cat:<10}: {np.mean(per_class_auc[idxs]):.4f}')


## Phase 1 Findings

**Baseline AUC (A0):** 0.XXXX on 200-window soundscape holdout

**Best Phase 1 AUC (C1):** 0.XXXX — combination of [list winning factors]

### Ablation Results

| Factor | AUC | Delta vs A0 | Notes |
|--------|-----|-------------|-------|
| A0: baseline | 0.XXXX | — | |
| A1: random crop | 0.XXXX | +/- 0.00 | |
| A2: specaug | 0.XXXX | +/- 0.00 | |
| A3: soundscape labels | 0.XXXX | +/- 0.00 | |
| A4: efficientnet_b1 | 0.XXXX | +/- 0.00 | |
| A5: cosine_warmup | 0.XXXX | +/- 0.00 | |
| A6: rating >= 3.0 | 0.XXXX | +/- 0.00 | |
| B1: mixup | 0.XXXX | +/- 0.00 | |
| B2: specaug+mixup | 0.XXXX | +/- 0.00 | |
| B3: SED head | 0.XXXX | +/- 0.00 | |
| B4: adamw wd=1e-4 | 0.XXXX | +/- 0.00 | |
| B5: focal loss | 0.XXXX | +/- 0.00 | |
| B6: secondary labels | 0.XXXX | +/- 0.00 | |
| B7: efficientnet_b2 | 0.XXXX | +/- 0.00 | |
| C1: best combo | 0.XXXX | +/- 0.00 | 20 epochs |

### Bottom 20 Species
*(paste output from D1 cell here)*

### Observations for Phase 2
- Fill in after seeing results


In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

sample_sub = pd.read_csv(SAMPLE_SUB)
test_files = sorted(TEST_DIR.glob('*.ogg'))
print(f'Test soundscapes : {len(test_files)}')

In [ ]:
from sklearn.metrics import roc_auc_score

model.eval()
all_probs, all_targets = [], []

for _, row in labels.iterrows():
    sc_path = TRAIN_SC_DIR / row['filename']
    if not sc_path.exists():
        continue
    start_sec = int(row['start'].split(':')[2]) + int(row['start'].split(':')[1]) * 60
    audio, _  = librosa.load(str(sc_path), sr=SR, offset=start_sec, duration=DURATION)
    if len(audio) < SR * DURATION:
        audio = np.pad(audio, (0, SR * DURATION - len(audio)))

    spec = audio_to_melspec(audio)
    spec = np.stack([spec, spec, spec], axis=0)
    spec = torch.tensor(spec).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = torch.sigmoid(model(spec)).cpu().numpy()[0]

    target = np.zeros(NUM_CLASSES, dtype=np.float32)
    for sp in row['primary_label'].split(';'):
        sp = sp.strip()
        if sp in species_to_idx:
            target[species_to_idx[sp]] = 1.0

    all_probs.append(probs)
    all_targets.append(target)

all_probs   = np.array(all_probs)
all_targets = np.array(all_targets)

# macro ROC-AUC skipping classes with no positive labels
scores = []
for i in range(NUM_CLASSES):
    if all_targets[:, i].sum() > 0:
        scores.append(roc_auc_score(all_targets[:, i], all_probs[:, i]))

print(f'Validation ROC-AUC: {np.mean(scores):.4f}')

In [ ]:
def predict_soundscape(filepath):
    audio, _  = librosa.load(str(filepath), sr=SR)
    chunk_len  = SR * DURATION
    results    = {}
    stem       = Path(filepath).stem

    for start in range(0, len(audio), chunk_len):
        chunk = audio[start:start + chunk_len]
        if len(chunk) < chunk_len:
            chunk = np.pad(chunk, (0, chunk_len - len(chunk)))

        spec  = audio_to_melspec(chunk)
        spec  = np.stack([spec, spec, spec], axis=0)
        spec  = torch.tensor(spec).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            probs = torch.sigmoid(model(spec)).cpu().numpy()[0]

        end_sec = (start + chunk_len) // SR
        results[f'{stem}_{end_sec}'] = probs

    return results


all_preds = {}
for fpath in tqdm(test_files, desc='Predicting'):
    all_preds.update(predict_soundscape(fpath))

print(f'Total prediction rows: {len(all_preds)}')